# 01 · Data index and leakage-safe split  *(shared foundation)*

**Purpose:** build the table of all images and create the **train / validation / test split** the whole team will use.
Create it **once**, commit `data/splits/`, and make sure Krishna and Rucha use the *same* files.

**Run order of Onkar's notebooks:** 01 → 02 baseline CNN → 03 MobileNetV2 → 04 MobileNetV2 experiments → 05 hand-off.
GPU is not needed here. Run the cells top to bottom (Runtime → Run all is fine).

In [ ]:
# ============ SETUP - run this first after every fresh Colab runtime ============
import os, sys, subprocess
from pathlib import Path

REPO_URL = ""   # e.g. "https://github.com/<your-username>/fruit-freshness-classifier.git"
                # leave "" if you uploaded/unzipped the project folder yourself

IN_COLAB = "google.colab" in sys.modules
PROJECT_DIR = Path("/content/fruit-freshness-classifier") if IN_COLAB else Path.cwd().parent
if not (PROJECT_DIR / "src").exists():
    if REPO_URL:
        subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)
    else:
        raise SystemExit(f"Project folder not found at {PROJECT_DIR}. Set REPO_URL above or upload the project there.")
elif REPO_URL and IN_COLAB:
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], check=False)

if os.getenv("FRUIT_SKIP_PIP") != "1":     # only kagglehub is needed; Colab already has TensorFlow etc.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_DIR / "requirements-colab.txt")], check=True)
sys.path.insert(0, str(PROJECT_DIR / "src"))

import json
import matplotlib.pyplot as plt
import pandas as pd
from fruitfresh import config
from fruitfresh.colab import setup_workspace, gpu_summary

WORK_DIR = setup_workspace()      # Colab: Google Drive (survives runtime resets)
SEED = config.SEED
print(gpu_summary())
print("Project folder :", PROJECT_DIR)
print("Work folder    :", WORK_DIR, "(models, histories, results are saved here)")

## 1 · Get the dataset
Downloads *Fruits Fresh and Rotten for Classification* from Kaggle (public, no login needed).

In [ ]:
from fruitfresh.data_index import get_dataset_root
DATASET_ROOT = get_dataset_root()
print("Dataset root:", DATASET_ROOT)

## 2 · Build the image index
Every file gets: class, **source photo** (`source_id`), original-or-augmented, and capture time.
Why this matters for this dataset:
* only ~1,500 *original* photos exist; most files are rotated / flipped / shifted / noisy **copies** (the prefix in the file name tells you which);
* the same original can sit in both Kaggle `train/` and `test/` folders (exact duplicates are removed);
* originals are screenshots whose file name contains the **capture time** - photos taken seconds apart usually show the same fruit.

In [ ]:
from fruitfresh.data_index import build_index, summarize_index
index = build_index(DATASET_ROOT)
summary = summarize_index(index)
print(json.dumps(summary, indent=2))
index.head()

**Check these numbers before moving on**
* `n_sources` is **much smaller** than `n_images` (about 1,500 vs 13,600 for the real dataset).
* `share_original_images` is around 0.1: roughly 9 out of 10 files are augmented copies.
* `sources_with_timestamp` equals `n_sources` (if not, those photos fall back to a random split).
* `capture_dates` shows only a few days: all photos come from a few capture sessions.

In [ ]:
display(pd.crosstab(index.class_name, index.is_original.map({True: "original", False: "augmented"}), margins=True))
display(pd.crosstab(index.class_name, index.kaggle_folder, margins=True))

In [ ]:
from fruitfresh.data_index import find_unreadable
bad = find_unreadable(index, DATASET_ROOT)
print("Unreadable (corrupt) images:", len(bad), bad[:5])

In [ ]:
from fruitfresh.viz import sample_grid
fig = sample_grid(index, DATASET_ROOT, per_class=4); plt.show()

## 3 · Create the leakage-safe split
**Time-block split** (details: `docs/TRAINING_GUIDE.md`):
1. all copies of one photo stay together (`source_id`);
2. per class, photos are sorted by capture time and cut into contiguous **blocks**;
3. whole blocks go to train / validation / test (≈70 / 15 / 15 %);
4. photos closer than **60 s** to a photo of a *different* split are dropped, so splits are separated by a real time gap.

`verify_split` **raises an error** if any rule is broken.

In [ ]:
from fruitfresh.splits import make_time_block_split, verify_split, save_split
index_split, split_report = make_time_block_split(index, seed=SEED, **config.SPLIT_PARAMS)
stats = verify_split(index_split, buffer_seconds=config.SPLIT_PARAMS["buffer_seconds"])
print(json.dumps({k: stats[k] for k in ("images", "sources", "n_sources_dropped", "min_cross_split_gap_seconds")}, indent=2))
display(pd.DataFrame(stats["images_per_class"]))

In [ ]:
print("Time blocks per class and split (few blocks = high variance for that class):")
display(pd.DataFrame(stats["blocks_per_class"]).T)
from fruitfresh.viz import split_timeline
fig = split_timeline(index_split); plt.show()

**Read the timeline:** each dot is one photo; colours must appear as separate stretches (not mixed), with grey dropped photos at the borders.
Some classes have only 1-3 test blocks, so a single unusual capture session can move that class's score - mention this in the report.

## 4 · Save the split (and commit it)

In [ ]:
import shutil
split_dir = WORK_DIR / "data" / "splits"
manifest = save_split(index_split, split_dir, {**config.SPLIT_PARAMS, "seed": SEED, **split_report}, stats)
repo_split = PROJECT_DIR / "data" / "splits"; repo_split.mkdir(parents=True, exist_ok=True)
for f in split_dir.glob("*"):
    shutil.copy(f, repo_split / f.name)
print("Saved to Drive :", split_dir)
print("Copied to repo :", repo_split, "-> commit these files")
print(json.dumps(manifest["files"], indent=2))

## Checklist before notebook 02
- [ ] `verify_split` passed and the timeline looks like separate stretches
- [ ] `data/splits/*.csv` and `split_manifest.json` are committed to GitHub
- [ ] Krishna and Rucha know **not to regenerate** the split (the SHA-256 in the manifest detects edited CSVs)